# AgentPulse | Predict AI agent failure
Zubair Hussain · MIT · Synthetic research demo, not production performance. Validation selects the threshold; test is held out. No internet or GPU required. Full-size data is sampled per shard to limit memory.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
FEATURES = ['elapsed_ms', 'input_tokens', 'output_tokens', 'tool_calls', 'retry_count', 'queue_ms', 'rate_limit_ratio', 'tool_error_ratio', 'context_utilization', 'cpu_utilization', 'memory_utilization', 'retrieval_score', 'budget_utilization', 'network_jitter_ms', 'concurrency', 'step_count']
base = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('data/demo')
assert base.exists(), 'Run from repository root or attach Kaggle dataset'
def read(split):
    paths = sorted([*base.rglob(split+'-*.csv'), *base.rglob(split+'-*.parquet')])
    assert paths, f'Missing {split} data'
    frames=[]
    for i,p in enumerate(paths):
        d=pd.read_csv(p) if p.suffix=='.csv' else pd.read_parquet(p)
        frames.append(d.sample(min(1000,len(d)),random_state=2026+i) if len(d)>1000 else d)
    return pd.concat(frames,ignore_index=True)
train,val,test=[read(s) for s in ('train','validation','test')]
assert not set(train.environment_id)&set(test.environment_id)
print([len(d) for d in (train,val,test)])


In [ ]:
scaler=StandardScaler().fit(train[FEATURES])
model=LogisticRegression(max_iter=300,random_state=2026).fit(scaler.transform(train[FEATURES]),train.failed)
pv=model.predict_proba(scaler.transform(val[FEATURES]))[:,1]
t=max(np.arange(.1,.81,.025),key=lambda a:f1_score(val.failed,pv>=a,zero_division=0))
p=model.predict_proba(scaler.transform(test[FEATURES]))[:,1]
print({'roc_auc':roc_auc_score(test.failed,p),'f1':f1_score(test.failed,p>=t),'pr_auc':average_precision_score(test.failed,p)})


## Next steps
Try group-aware validation, calibration and real traces with consent. Do not present simulated results as real-world performance.
